[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/kerr/kerr_comparison.ipynb)

# Kerr: Metric and Gauge Fields Side by Side

One rotating black hole, one camera, two descriptions of the same light. The ray pictures should coincide, and a polarizer should read the same transported light. The metric description carries coordinate screens; the GTG description carries local screens. Each standalone notebook develops the same scenes: [GTG](kerr.ipynb) and [metric](kerr_metric.ipynb).

Both descriptions distinguish a direction from the covector that reads it. Their difference is where gravitational geometry enters: a metric and its Levi-Civita connection, or a frame map and a bivector connection. The source and observer choices are held fixed.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga@main"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from dataclasses import dataclass

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA as ga
from examples.util.animation import save_figure, show_animation
from examples.relativity.kerr import render
from examples.relativity.kerr.core import resolve, trace

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Events and directions are antivectors; momenta, gradients and the slices that read coordinates are
# vectors, covectors that pair with directions.
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Antivector = ga.gatype.antivector()
# The hole's spin per unit mass: intrinsic, the patch of the plane it turns in, a plane through time.
Spin = Bivector.derive.containing("t").derive.flipped_orientation
Rotor = ga.gatype.rotor()
# Directions on the camera's screen, along its own x and y: the antivectors that hold both t and z.
Screen = Antivector.derive.containing("t z")
# The camera's slices x and y, which read screen directions: vectors, extrinsic.
ScreenSlice = Vector.derive.within("x y")
ScreenFrame = ga.gatype((Antivector, Screen))                # Antivector <- Screen
# The light's polarization covariance, from the slices that read the screen to screen directions.
Coherency = ga.gatype((Screen, ScreenSlice))                 # Screen <- ScreenSlice
# The position gauge field: from a direction of the local frame to one of the Kerr geometry.
GaugeField = ga.gatype((Antivector, Antivector))             # Antivector <- Antivector
# The null direction's derivative: from a step to the direction's change along it.
DirectionGradient = ga.gatype((Antivector, Antivector))      # Antivector <- Antivector
# One unit along the lab's time, the direction across the lab's slice of space `mv.t`, and the part of a
# direction within that slice.
time = -mv.t.dual()                                          # [] Antivector
space = Antivector - time * (mv.t & Antivector)              # Antivector <- Antivector
# One unit along time and along each axis of space: an event's coordinates weight them.
axes = stack([time, mv.x.dual(), mv.y.dual(), mv.z.dual()])  # [4] Antivector
# G = c = M = 1.
mass = 1.0
Metric = ga.gatype((Vector, Antivector))
InverseMetric = ga.gatype((Antivector, Vector))

## 1. One geometry, two primary maps

| Operation | GTG | Metric |
| --- | --- | --- |
| Gravitational field | `gauge(geometry)`, direction to direction | `metric(geometry)`, direction to covector |
| Squared proper time | `-gauge_inverse(geometry)(u).squared()` | `metric(geometry)(u) & u` |
| Ray Hamiltonian | `-local(geometry, p).squared() / 2` | `(p & inverse_metric(geometry)(p)) / 2` |
| Tangent | `gauge(geometry)(local(geometry, p))` | `inverse_metric(geometry)(p)` |
| Screen transport | local screen, bivector commutator | coordinate screen, Christoffel connection |

The null dyad squares to zero. Consequently `gauge_inverse` changes its sign, and applying `gauge` twice doubles it. This factorizes the metric. It does not determine the Kerr profile: that is supplied by the gravitational solution.

In matrix notation, writing $E$ for `gauge`, $g=E^{-\mathsf T}\eta E^{-1}$ and $g^{-1}=E\eta^{-1}E^{\mathsf T}$. The use of $E$ twice is specific to this flat-metric-self-adjoint Kerr–Schild frame. A general frame uses its dual map on covectors.

In [ ]:
@dataclass(frozen=True)
class Field:
    radius: Scalar
    profile: Scalar
    null: Antivector
    profile_gradient: Vector
    null_gradient: DirectionGradient
    null_curl: Bivector

    @property
    def null_plane(self) -> Vector:
        """The null direction's plane, the light cone's tangent plane along it, which contains the
        direction: minus its dual."""
        return -self.null.dual()                                                            # [...] Vector


def field(position: Antivector, spin: Spin, mass: float) -> Field:
    """The Kerr profile, null direction and their derivatives at a batch of events."""
    spatial = space(position)                                                               # [...] Antivector
    # The plane the hole turns in, whose dual is its spin.
    turning = spin.dual_inverse()                                                           # [...] Bivector
    # The volume the position spans with the turning plane: the plane times the position's height off it,
    # held by its complement along time. Wedged with the plane it is that height along the spin, times the
    # spin squared; the plane enters twice, so its sense of turning drops out.
    volume = spatial.anticommutator(turning)                                                # [...] Vector
    height = volume ^ turning                                                               # [...] Antivector
    spin_squared = spin.squared()                                                           # [...] Scalar
    offset = spatial.squared() - spin_squared                                               # [...] Scalar
    discriminant = (offset.squared() - 4 * volume.squared()).square_root()                  # [...] Scalar
    radius_squared = (offset + discriminant) / 2                                            # [...] Scalar
    radius = radius_squared.square_root()                                                   # [...] Scalar
    # Paired with a small step, how far the step changes the oblate radius.
    radius_gradient = (radius_squared * spatial - height).dual() / (radius * discriminant)  # [...] Vector
    # The hole turns as `(turning * (-angle / 2)).exp()` does, and drags the frame with it: the outgoing
    # direction twists against the turning, by the position's commutator with the plane.
    direction = (radius * spatial - spatial.commutator(turning) - height / radius) / (radius_squared + spin_squared)  # [...] Antivector
    lever = spatial - 2 * radius * direction + height / radius_squared                      # [...] Antivector
    direction_gradient = (
        lever * (radius_gradient & Antivector)
        + radius * space - space.commutator(turning)
        - (space.anticommutator(turning) ^ turning) / radius
    ) / (radius_squared + spin_squared)                                                     # [...] Antivector <- Antivector
    denominator = radius_squared.squared() - volume.squared()                               # [...] Scalar
    profile = mass * radius**3 / denominator                                                # [...] Scalar
    profile_gradient = profile * (3 * radius_gradient / radius
        - (4 * radius**3 * radius_gradient - 2 * height.dual()) / denominator)              # [...] Vector
    # The curl of the null covector, `-(Vector ^ null_gradient(Antivector).dual()).trace(1, 2)`, piece by
    # piece: the rank-one piece gives its two covectors wedged, the twist by the plane twice the plane, the
    # rest none.
    null_curl = ((radius_gradient ^ lever.dual()) - 2 * turning) / (radius_squared + spin_squared)  # [...] Bivector
    return Field(radius, profile, time - direction, profile_gradient, -direction_gradient, null_curl)  # [...] Field


def dyad(geometry: Field) -> GaugeField:
    """The null dyad, weighted by the profile: the null direction, read by its null plane."""
    return geometry.profile * geometry.null * (geometry.null_plane & Antivector)            # [...] Antivector <- Antivector


def gauge(geometry: Field) -> GaugeField:
    """The position gauge field: the identity plus the weighted null dyad, from a direction of the local
    frame to one of the Kerr geometry."""
    return Antivector + dyad(geometry)                                                      # [...] Antivector <- Antivector


def gauge_inverse(geometry: Field) -> GaugeField:
    """The gauge field's inverse: the dyad squares to zero, so the identity less the dyad."""
    return Antivector - dyad(geometry)                                                      # [...] Antivector <- Antivector


def local(geometry: Field, momentum: Vector) -> Antivector:
    """A covector momentum's direction in the local frame: raised flat, `-momentum.dual()`, and carried
    through the gauge field. Minus half its flat square is the ray's Hamiltonian."""
    return gauge(geometry)(-momentum.dual())                                                # [...] Antivector


def proper_squared(geometry: Field, direction: Antivector) -> Scalar:
    """The squared proper length of a timelike direction: minus the flat square of the direction carried
    back into the local frame."""
    return -gauge_inverse(geometry)(direction).squared()                                    # [...] Scalar


def overlap_gradient(geometry: Field, momentum: Vector) -> Vector:
    """The gradient of the momentum's pairing with the null direction, `momentum & null`: the momentum
    pulled back through the null direction's derivative."""
    return geometry.null_gradient.adjugate()(momentum)                                      # [...] Vector
def metric(geometry: Field) -> Metric:
    """The physical metric: a direction's covector, with positive timelike square."""
    plane = geometry.null_plane
    return -Antivector.dual() - 2 * geometry.profile * plane * (plane & Antivector)


def inverse_metric(geometry: Field) -> InverseMetric:
    """The inverse metric, taking a covector momentum to its coordinate tangent."""
    null = geometry.null
    return -Vector.dual() + 2 * geometry.profile * null * (Vector & null)

## 2. The same Hamiltonian rays

The momentum's reading on its tangent gives twice the Hamiltonian in either description. Its position derivative is the same profile gradient and pullback through the null direction's derivative. The positive-time physical metric and the algebra's negative time square account for the minus sign in the GTG expression.

In [ ]:
def hamiltonian(geometry: Field, momentum: Vector) -> Scalar:
    """The ray Hamiltonian, positive for a timelike momentum in the physical +--- convention."""
    return -local(geometry, momentum).squared() / 2                                         # [...] Scalar


def motion(geometry: Field, momentum: Vector) -> tuple[Antivector, Vector]:
    """The two Hamiltonian derivatives at a field: the ray tangent and minus its position gradient."""
    overlap = momentum & geometry.null                                                      # [...] Scalar
    # Minus half the flat square of the momentum's local direction is the Hamiltonian; its momentum derivative
    # is that direction carried through the gauge field once more.
    tangent = gauge(geometry)(local(geometry, momentum))                                    # [...] Antivector
    # The gradient of profile * overlap squared, by the product rule.
    gradient = (geometry.profile_gradient * overlap.squared()
                + 2 * geometry.profile * overlap * overlap_gradient(geometry, momentum))      # [...] Vector
    return tangent, -gradient                                                               # [...] Antivector, [...] Vector


def rates(position: Antivector, momentum: Vector, spin: Spin, mass: float) -> tuple[Antivector, Vector]:
    """The two Hamiltonian derivatives: ray tangent and minus its position gradient."""
    return motion(field(position, spin, mass), momentum)                                    # [...] Antivector, [...] Vector


def rotation(geometry: Field, direction: Antivector) -> Bivector:
    """The rotation gauge field: the plane the local frame turns or boosts in, per unit step along a
    direction."""
    # The null plane's reading of the direction.
    null = geometry.null_plane                                                              # [...] Vector
    along = null & direction                                                                # [...] Scalar
    # The gradient of profile * along.
    turning = geometry.profile_gradient * along + geometry.profile * overlap_gradient(geometry, -direction.dual())  # [...] Vector
    return -(turning ^ null) - geometry.profile * geometry.null_curl * along                # [...] Bivector


def polarized_rates(position: Antivector, momentum: Vector, screen: ScreenFrame,
                    spin: Spin, mass: float) -> tuple[Antivector, Vector, ScreenFrame]:
    """Hamiltonian ray motion and parallel transport of its camera screen, per unit of decreasing
    coordinate time."""
    geometry = field(position, spin, mass)                                                  # [...] Field
    tangent, momentum_rate = motion(geometry, momentum)                                     # [...] Antivector, [...] Vector
    screen_rate = -rotation(geometry, local(geometry, momentum)).commutator(screen)         # [...] Antivector <- Screen
    clock = -(mv.t & tangent)                                                               # [...] Scalar
    return tangent / clock, momentum_rate / clock, screen_rate / clock                      # [...] Antivector, [...] Vector, [...] Antivector <- Screen

In [ ]:
def metric_hamiltonian(geometry: Field, momentum: Vector) -> Scalar:
    """Half the momentum's reading of its image under the inverse metric."""
    return (momentum & inverse_metric(geometry)(momentum)) / 2


def metric_motion(geometry: Field, momentum: Vector) -> tuple[Antivector, Vector]:
    """The two Hamiltonian derivatives at a field: the ray tangent and minus its position gradient."""
    overlap = momentum & geometry.null                                                      # [...] Scalar
    # Differentiating the quadratic Hamiltonian with respect to momentum applies the inverse metric.
    tangent = inverse_metric(geometry)(momentum)                                    # [...] Antivector
    # The gradient of profile * overlap squared, by the product rule.
    gradient = (geometry.profile_gradient * overlap.squared()
                + 2 * geometry.profile * overlap * overlap_gradient(geometry, momentum))      # [...] Vector
    return tangent, -gradient                                                               # [...] Antivector, [...] Vector


def metric_rates(position: Antivector, momentum: Vector, spin: Spin, mass: float) -> tuple[Antivector, Vector]:
    """The two Hamiltonian derivatives: ray tangent and minus its position gradient."""
    return metric_motion(field(position, spin, mass), momentum)                                    # [...] Antivector, [...] Vector


def metric_change(geometry: Field, along: Antivector, direction: Antivector) -> Vector:
    """The metric's directional derivative along a coordinate step, applied to a fixed direction."""
    plane = geometry.null_plane
    plane_change = -geometry.null_gradient(along).dual()
    return -2 * (
        (geometry.profile_gradient & along) * plane * (plane & direction)
        + geometry.profile * (plane_change * (plane & direction)
                              + plane * (plane_change & direction)))


def christoffel(geometry: Field, along: Antivector, direction: Antivector) -> Antivector:
    """The torsion-free metric connection in the fixed Cartesian coordinate basis."""
    plane = geometry.null_plane
    along_reading, direction_reading = plane & along, plane & direction
    along_gradient = geometry.null_gradient.adjugate()(-along.dual())
    direction_gradient = geometry.null_gradient.adjugate()(-direction.dual())
    # Differentiate the scalar metric reading with both coordinate directions held fixed.
    reading_gradient = -2 * (
        geometry.profile_gradient * along_reading * direction_reading
        + geometry.profile * (along_gradient * direction_reading
                              + along_reading * direction_gradient))
    # Two directional derivatives and the scalar reading's gradient give the lowered connection;
    # the inverse metric returns its direction.
    return inverse_metric(geometry)(
        metric_change(geometry, along, direction)
        + metric_change(geometry, direction, along) - reading_gradient) / 2


def metric_polarized_rates(position: Antivector, momentum: Vector, screen: ScreenFrame,
                    spin: Spin, mass: float) -> tuple[Antivector, Vector, ScreenFrame]:
    """Hamiltonian metric_motion and coordinate-screen transport, per unit of decreasing coordinate time."""
    geometry = field(position, spin, mass)
    tangent, momentum_rate = metric_motion(geometry, momentum)
    screen_rate = -christoffel(geometry, tangent, screen)
    clock = -(mv.t & tangent)
    return tangent / clock, momentum_rate / clock, screen_rate / clock

In [ ]:
spin = 0.8 * mv.xy.dual()
extent = 12.0 * mass
n_rays, n_steps = 31, 400
offsets = np.linspace(-8, 8, n_rays) * mass
step_size, stop_fraction = 0.1 * mass, 0.8
n_ring_samples = 128
position = -mv.x.dual() * extent + mv.y.dual() * offsets
geometry = field(position, spin, mass)
local_null = time + mv.x.dual()
momentum = -gauge_inverse(geometry)(local_null).dual()
momentum = momentum / (momentum & time)
position = position.broadcast_to(momentum.shape)

In [ ]:
gtg_paths, gtg_momenta = trace(position, momentum, spin, mass, step_size, n_steps, stop_fraction, rates)
metric_paths, metric_momenta = trace(position, momentum, spin, mass, step_size, n_steps, stop_fraction, metric_rates)
horizon_radius = mass + (mass**2 - spin.squared()).square_root()
ring_radius = (horizon_radius.squared() + spin.squared()).square_root()
angles = np.linspace(0, 2 * np.pi, n_ring_samples + 1)
horizon = ((mv.xy * (-angles / 2)).exp() >> mv.x.dual()) * ring_radius
save_figure(render.draw_ray_comparison((gtg_paths, metric_paths), horizon, extent), "kerr_comparison_rays");

## 3. One physical screen in two representations

The same screen is `coordinate_screen = gauge(geometry)(local_screen)`. Its two derivatives differ because the frame varies along the ray. The metric connection carries the coordinate screen; the bivector connection carries the local one. Both preserve the physical screen's lengths and transversality.

In tensor-index notation, $S=Es$ and $\dot S=(\dot E)s+E\dot s=-\Gamma(\dot x)S$. The frame-derivative term is essential when comparing the two rates. Finite Runge–Kutta steps in the two screen variables need not agree exactly; their difference decreases with the step size.

In [ ]:
def orbiting_gas(position: Antivector, orbit: Bivector, spin: Spin, mass: float) -> Antivector:
    """The unit velocity of gas on a circular equatorial orbit in the unit plane `orbit`, at each event."""
    geometry = field(position, spin, mass)                                                  # [...] Field
    angular_speed = orbital_speed(geometry.radius, orbit, spin, mass)                       # [...] Scalar
    # A point turned by `(orbit * (-angle / 2)).exp()` moves by its commutator with the orbit's plane.
    flow = time + angular_speed * space(position).commutator(orbit)                         # [...] Antivector
    return flow / proper_squared(geometry, flow).square_root()                              # [...] Antivector


def orbital_speed(radius: Scalar, orbit: Bivector, spin: Spin, mass: float) -> Scalar:
    """Kepler's angular speed, per coordinate time, of a circular equatorial orbit in the unit plane
    `orbit` at each oblate radius, the spin counted along the orbit."""
    alignment = orbit & spin                                                                # [] Scalar
    return np.sqrt(mass) / (radius * radius.square_root() + alignment * np.sqrt(mass))      # [...] Scalar


def frequency_ratio(position: Antivector, momentum: Vector, camera: Antivector, orbit: Bivector,
                    spin: Spin, mass: float) -> Scalar:
    """The frequency a camera at rest at `camera` sees, over the one emitted by gas on a circular
    orbit in the unit plane `orbit`, for rays whose momenta are given at the gas."""
    gas = orbiting_gas(position, orbit, spin, mass)                                         # [...] Antivector
    at_rest = time / proper_squared(field(camera, spin, mass), time).square_root()         # [] Antivector
    # The momentum paired with time is conserved along the ray, so the camera's frequency reads here.
    return (momentum & at_rest) / (momentum & gas)                                          # [...] Scalar


def cube_root(value: Scalar) -> Scalar:
    """The real cube root of each non-negative scalar."""
    return (value.log() / 3).exp()


def innermost_stable_orbit(orbit: Bivector, spin: Spin, mass: float) -> Scalar:
    """The radius of the innermost stable circular equatorial orbit in the unit plane `orbit`, turning
    with the hole, the spin counted along the orbit: 6M without spin, M at the extreme (Bardeen, Press
    and Teukolsky, 1972)."""
    ratio = (orbit & spin) / mass                                                           # [] Scalar
    first = 1 + cube_root(1 - ratio**2) * (cube_root(1 + ratio) + cube_root(1 - ratio))     # [] Scalar
    second = (3 * ratio**2 + first**2).square_root()                                        # [] Scalar
    return mass * (3 + second - ((3 - first) * (3 + first + 2 * second)).square_root())    # [] Scalar


def disk_temperature(radius: Scalar, orbit: Bivector, spin: Spin, mass: float, scale: float) -> Scalar:
    """A relativistic thin disk's temperature at each oblate radius, from the flux of Page and Thorne
    (1974), for gas orbiting in the unit plane `orbit`, the spin counted along the orbit: zero at the
    innermost stable orbit, falling as the radius to the power -3/4 far out. `scale` is
    (accretion rate c^6 / (σ G^2 M^2))^(1/4), K."""
    ratio = (orbit & spin) / mass                                                           # [] Scalar
    root_radius = (radius / mass).square_root()                                             # [...] Scalar
    edge = (innermost_stable_orbit(orbit, spin, mass) / mass).square_root()                 # [] Scalar
    # The roots of the cubic `root_radius**3 - 3 * root_radius + 2 * ratio`, and the weight of each in
    # the torque; the product of a root's differences from the other two is the cubic's derivative there.
    roots = 2 * ((ratio.arccos() + np.array([-np.pi, np.pi, 3 * np.pi])) / 3).cos()       # [3] Scalar
    weights = (roots - ratio) ** 2 / (roots * (roots ** 2 - 1))                             # [3] Scalar
    # The torque the disk carries, zero at the innermost stable orbit; round-off kept non-negative.
    torque = (root_radius - edge - 1.5 * ratio * (root_radius / edge).log()
              - (weights * ((root_radius[..., None] - roots) / (edge - roots)).log()).sum(axis=-1)).clip(0, np.inf)  # [...] Scalar
    flux = 3 / (8 * np.pi) * torque / (root_radius ** 4 * (root_radius ** 3 - 3 * root_radius + 2 * ratio))  # [...] Scalar
    return scale * flux.square_root().square_root()                                         # [...] Scalar


def disk_depth(radius: Scalar, inner_radius: Scalar, outer_radius: float, depth: float) -> Scalar:
    """The disk's optical depth straight through its thickness at each oblate radius: `depth` at the
    inner edge, where the gas is densest, thinning outwards to none at the outer edge."""
    fraction = (radius - inner_radius) / (outer_radius - inner_radius)                      # [...] Scalar
    return depth * (1 - fraction) * (1 - fraction)                                          # [...] Scalar


def disk_opacity(position: Antivector, momentum: Vector, depth: Scalar, normal: Vector,
                 orbit: Bivector, spin: Spin, mass: float) -> Scalar:
    """The share of a ray's light the disk absorbs where the ray crosses it. Its optical depth along
    the ray is the depth through its thickness over the cosine of the ray's slant as the orbiting gas
    sees it: the ray's energy in the gas's frame over its momentum along the normal. Gas moving away
    from the light's source sees the ray more obliquely, and absorbs more."""
    gas = orbiting_gas(position, orbit, spin, mass)                                         # [...] Antivector
    slant = ((momentum & gas) / (momentum & normal.dual())).abs()                           # [...] Scalar
    return 1 - (-depth * slant).exp()                                                       # [...] Scalar


def camera_boost(geometry: Field) -> Rotor:
    """A coordinate-static camera's rest frame in the Kerr–Schild frame, outside the static limit."""
    observer = gauge_inverse(geometry)(time) / proper_squared(geometry, time).square_root()  # [...] Antivector
    return (1 - observer * time).normalized()                                               # [...] Rotor


def initial_momentum(position: Antivector, direction: Antivector, spin: Spin, mass: float) -> Vector:
    """A future null covector with the given unit spatial direction in the Kerr–Schild frame."""
    geometry = field(position, spin, mass)                                                  # [...] Field
    local_null = time + direction                                                           # [...] Antivector
    # Carried back out of the local frame and lowered flat.
    momentum = -gauge_inverse(geometry)(local_null).dual()                                  # [...] Vector
    return momentum / (momentum & time)                                                     # [...] Vector


def camera_rays(eye: Antivector, orientation: Rotor, pixel_width: int, pixel_height: int,
                half_view: float, spin: Spin, mass: float) -> tuple[Antivector, Vector]:
    """Pixel-centre rays of a coordinate-static camera, looking along its -z, with right +y and up -x."""
    horizontal = ((np.arange(pixel_width) + 0.5) / pixel_width * 2 - 1) * half_view * pixel_width / pixel_height
    vertical = (1 - (np.arange(pixel_height) + 0.5) / pixel_height * 2) * half_view
    x, y, z = mv.x.dual(), mv.y.dual(), mv.z.dual()
    direction = orientation >> (-z + y * horizontal[None, :] - x * vertical[:, None])
    direction = direction / direction.squared().square_root()
    geometry = field(eye, spin, mass)
    # The future photon arrives opposite the direction the camera looks; its rest frame is boosted
    # into the Kerr–Schild frame before its heading is lowered into a coordinate momentum.
    heading = camera_boost(geometry) >> (time - direction)
    momentum = -gauge_inverse(geometry)(heading).dual()
    momentum = -momentum / (momentum & time)
    momentum = momentum.reshape(-1)
    # Every ray starts at the eye, as an event of the whole of spacetime, the type its steps carry.
    return eye.broadcast_to(momentum.shape), momentum


def camera_screen(position: Antivector, momentum: Vector, orientation: Rotor,
                  spin: Spin, mass: float) -> ScreenFrame:
    """Orthonormal transverse camera axes, with horizontal x and vertical y screen inputs."""
    geometry = field(position, spin, mass)
    boost = camera_boost(geometry)
    heading = boost.reverse() >> local(geometry, momentum)                                 # [...] Antivector
    direction = space(heading) / -(mv.t & heading)                                          # [...] Antivector
    # The camera's own right and up, local +y and -x, each less its parts along the directions before
    # it; every direction here is a unit of space, squaring to one.
    right = orientation >> mv.y.dual()
    right = right - direction * (direction | right)
    right = right / right.squared().square_root()
    up = orientation >> -mv.x.dual()
    up = up - direction * (direction | up) - right * (right | up)
    up = up / up.squared().square_root()
    # A screen direction is read by pairing with the camera's slices x and y.
    return boost >> (right * (mv.x & Screen) + up * (mv.y & Screen))                         # [...] Antivector <- Screen


def disk_coherency(position: Antivector, momentum: Vector, screen: ScreenFrame,
                  normal: Vector, spin: Spin, mass: float, polarization_degree: float) -> Coherency:
    """Unit-intensity disk emission resolved along the parallel-transported camera axes; `normal` is the
    disk's plane, the covector that measures height above it."""
    heading = local(field(position, spin, mass), momentum)                                  # [...] Antivector
    direction = space(heading) / (mv.t & heading)                                           # [...] Antivector
    # The magnetic direction circles the normal, square to the plane through the normal and the
    # emitter. Emission is polarized perpendicular to it across the ray: in that plane, square to the ray.
    meridian = space(position).dual() ^ normal                                              # [...] Bivector
    polarization_axis = meridian.commutator(direction) / meridian.norm()                    # [...] Antivector
    polarization = screen.adjoint()(polarization_axis)                                      # [...] Screen
    # Polarization vanishes smoothly when looking along the magnetic direction.
    polarized_fraction = polarization_degree * polarization.squared()                       # [...] Scalar
    # Unpolarized light is round, where the metric enters; the polarized part weights the polarization by
    # each slice's reading of it.
    return ((1 - polarized_fraction) / 2 * ScreenSlice.dual()
            + polarization_degree * polarization * (ScreenSlice & polarization))            # [...] Screen <- ScreenSlice


def transmitted(coherency: Coherency, analyzers: ScreenSlice) -> Scalar:
    """The power each unit analyzer passes, an analyzer being the slice that reads the screen across its
    axis's normal: its reading of the coherency's image of itself."""
    return analyzers & coherency(analyzers)                                                 # [...] Scalar

## 4. The same camera and emitting disk

The camera is coordinate-static. Its angles, screen and frequency measurement use that same observer. The disk is prograde and subextremal; the magnetic pattern is prescribed in the Kerr–Schild source frame in both descriptions. Local observer frames can specify measurements in classical metric GR too; they do not require evolving the screen through a rotation gauge field.

In [ ]:
camera_spin = 0.8 * mass
camera_distance = 28.0 * mass
inclination = np.deg2rad(75.0)
half_view = np.tan(np.deg2rad(19.0))
image_width = 240
image_height = image_width * 2 // 3
step_fraction, n_camera_steps = 0.1, 720
outer_radius, inner_depth = 12.0 * mass, 4.0
temperature_scale, exposure = 58500.0, 6500.0
escape_radius = 100.0 * mass
capture_radius = 1.05 * (mass + np.sqrt(mass**2 - camera_spin**2))
sky_width, sky_seed = 1024, 8
sky_height = sky_width // 2
orientation = (mv.zx * (-inclination / 2)).exp()
eye = (orientation >> mv.z.dual()) * camera_distance
spin = mv.xy.dual() * camera_spin
inner_radius = innermost_stable_orbit(mv.xy, spin, mass)
position, momentum = camera_rays(eye, orientation, image_width, image_height, half_view, spin, mass)
local_screen = camera_screen(position, momentum, orientation, spin, mass)
coordinate_screen = gauge(field(position, spin, mass))(local_screen)


def opacity(event: Antivector, momentum: Vector, radius: Scalar) -> Scalar:
    depth = disk_depth(radius, inner_radius, outer_radius, inner_depth)
    return disk_opacity(event, momentum, depth, mv.z, mv.xy, spin, mass)

In [ ]:
gtg_image, gtg_end = resolve((position, momentum, local_screen), spin, mass, step_fraction, n_camera_steps,
    polarized_rates, mv.z, inner_radius, outer_radius, opacity, escape_radius, capture_radius)
metric_image, metric_end = resolve((position, momentum, coordinate_screen), spin, mass, step_fraction, n_camera_steps,
    metric_polarized_rates, mv.z, inner_radius, outer_radius, opacity, escape_radius, capture_radius)
gtg_points, gtg_momenta, gtg_screens = gtg_image.disk
metric_points, metric_momenta, metric_screens = metric_image.disk
gtg_temperature = frequency_ratio(gtg_points, gtg_momenta, eye, mv.xy, spin, mass) * disk_temperature(
    gtg_image.disk_radii, mv.xy, spin, mass, temperature_scale)
metric_temperature = frequency_ratio(metric_points, metric_momenta, eye, mv.xy, spin, mass) * disk_temperature(
    metric_image.disk_radii, mv.xy, spin, mass, temperature_scale)
sky = render.star_texture(sky_width, sky_height, sky_seed)
save_figure(render.draw_camera_comparison((gtg_image, metric_image), (gtg_temperature, metric_temperature),
    exposure, (image_height, image_width), sky), "kerr_comparison_camera");

## 5. The same rotating analyzer

GTG is on the left, metric transport on the right. Both screens are resolved in the specified source frame before reading the same polarization covariance. The covariance maps an analyzer slice to a screen direction; `analyzer & coherency(analyzer)` is its transmitted power. The metric enters the isotropic unpolarized contribution explicitly.

In [ ]:
n_polarizer_frames, duration_ms = 40, 100
polarization_degree = 0.8
polarizer_angles = np.linspace(0, np.pi, n_polarizer_frames, endpoint=False)
analyzers = (mv.xy * (-polarizer_angles / 2)).exp() >> mv.x
gtg_coherency = disk_coherency(gtg_points, gtg_momenta, gtg_screens, mv.z, spin, mass, polarization_degree)
metric_local_screens = gauge_inverse(field(metric_points, spin, mass))(metric_screens)
metric_coherency = disk_coherency(metric_points, metric_momenta, metric_local_screens, mv.z, spin, mass, polarization_degree)
gtg_transmission = transmitted(gtg_coherency, analyzers[:, None])
metric_transmission = transmitted(metric_coherency, analyzers[:, None])
show_animation(render.animate_polarizer_comparison((gtg_image, metric_image), (gtg_temperature, metric_temperature),
    exposure, (gtg_transmission, metric_transmission), (image_height, image_width), sky),
    "kerr_comparison_polarizer", duration_ms)

## 6. What the agreement establishes

Complementarity separates a direction from its reading in both formulations. The metric is a map between those types; the frame is a transformation between local and coordinate directions. The null-dyad factorization connects them, while the connection accounts for the frame's variation. This is an equivalence for the supplied Kerr solution, not a derivation of a gravitational action or field equation from complementarity.

The metric dependence has identifiable places: the physical response, normalized observers, and the connection chosen for parallel transport. Pulling a measurement back through a supplied map is metric-independent; choosing the Levi-Civita transport is a metric-dependent physical choice. [Francis and Kosowsky](https://arxiv.org/pdf/gr-qc/0311007) develop both the metric and bivector-connection descriptions.

In [ ]:
end_geometry = field(gtg_end[0], spin, mass)
coordinate_gtg_end_screen = gauge(end_geometry)(gtg_end[2])
readout = render.comparison_readout(gtg_end[0] - metric_end[0], gtg_end[1] - metric_end[1],
    coordinate_gtg_end_screen - metric_end[2],
    metric(end_geometry)(inverse_metric(end_geometry)(gtg_end[1])) - gtg_end[1])
readout

In [ ]:
# checks
# The two Hamiltonians and ray equations agree; both versions use the same incoming momenta.
np.testing.assert_allclose((hamiltonian(field(position, spin, mass), momentum) - metric_hamiltonian(field(position, spin, mass), momentum)).kernel,
                           0, atol=1e-11, rtol=0)
np.testing.assert_allclose((gtg_paths - metric_paths).kernel, 0, atol=1e-8, rtol=0)
np.testing.assert_array_equal(gtg_image.disk_pixels, metric_image.disk_pixels)
np.testing.assert_array_equal(gtg_image.sky_pixels, metric_image.sky_pixels)
# The camera images agree before polarization; different screen variables carry finite-step error.
np.testing.assert_allclose((gtg_temperature - metric_temperature).kernel, 0, atol=1e-4, rtol=0)